# Retos — Workshop RAG (hasta 30 puntos)

> **Workshop RAG · SI3003 · Juan José Palacio**

Este notebook es autosuficiente y continúa el sistema de la Parte B. Resuelve
cuatro retos (tope 30 pts; cada reto ~10 pts, se eligen los que se quieran):

| Reto | Título | Qué mide |
|---|---|---|
| **C1** | **FAISS vs HNSW** | Compromiso velocidad/precisión: medir el efecto de `ef` sobre el tiempo de consulta y el recall. |
| **C2** | **Tercer paper** | Añadir un paper más al índice y hacer una pregunta que lo conecte con los otros dos. |
| **C3** | **Filtros por metadata** | Restringir la búsqueda por `source`/`title` usando el payload de Qdrant. |
| **C4** | **Evaluación automática** | Métrica automática (keyword-recall) del RAG contra un *ground truth*. |

> En Colab ejecuta en orden. Requiere `NVIDIA_API_KEY` (celda de configuración).


## 0. Instalación y configuración

In [ ]:
%pip install -qU "markitdown[pdf]" faiss-cpu langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client langchain-community python-dotenv requests numpy


In [ ]:
import os, getpass, time
from pathlib import Path
from typing import List
import numpy as np
import requests
from dotenv import load_dotenv

load_dotenv()
try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Crea una en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
# --- Seleccion robusta del modelo de generacion (evita modelos deprecados) ---
# El modelo nvidia/nemotron-3-super-120b-a12b fue deprecado el 19/10/2026.
# Elegimos automaticamente el primer modelo de chat disponible en tu cuenta,
# priorizando los Nemotron mas recientes. Asi el notebook no se rompe si NVIDIA
# renombra o retira un modelo.
GENERATION_CANDIDATES = [
    "nvidia/nemotron-3.5-super-vl-120b-a12b",
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "nvidia/llama-3.3-nemotron-super-49b-v1",
    "nvidia/llama-3.1-nemotron-70b-instruct",
    "meta/llama-3.3-70b-instruct",
    "meta/llama-3.1-70b-instruct",
]

def pick_generation_model(candidates):
    from langchain_nvidia_ai_endpoints import ChatNVIDIA as _ChatNVIDIA
    try:
        available = {m.id for m in _ChatNVIDIA.get_available_models()}
    except Exception as exc:
        print(f"! No se pudo listar modelos ({exc}); uso el primer candidato.")
        return candidates[0]
    for name in candidates:
        if name in available:
            return name
    # Si ninguno coincide, toma cualquier modelo que parezca de chat/instruct.
    for m in sorted(available):
        if any(t in m for t in ("instruct", "nemotron", "chat")):
            return m
    raise RuntimeError("No hay modelos de generacion disponibles en tu cuenta NVIDIA.")

GENERATION_MODEL = pick_generation_model(GENERATION_CANDIDATES)
print(f"Modelo de generacion seleccionado: {GENERATION_MODEL}")
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
print("OK: NVIDIA NIM configurado")

## 1. Descargar y convertir 3 papers (base para C1–C4)

Reutilizo los dos papers de la Parte B y agrego un **tercer paper** (C2):
*BERT* (Devlin et al., 2018), muy citado por el paper de RAG (usa encoders BERT
en su retriever DPR), lo que permite preguntas que conectan los tres.


In [ ]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
    "bert_paper.pdf": "https://arxiv.org/pdf/1810.04805",   # C2 — tercer paper
}

def download_file(url, dest):
    if dest.exists() and dest.stat().st_size > 10_000:
        print(f"-> ya existe: {dest.name}"); return
    r = requests.get(url, timeout=90); r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError(f"{url} no es un PDF valido")
    dest.write_bytes(r.content)
    print(f"descargado: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")

for fn, url in PAPERS.items():
    download_file(url, DOCS_DIR / fn)

md_conv = MarkItDown(enable_plugins=False)
def load_pdf(path):
    res = md_conv.convert(str(path))
    if not res.text_content.strip():
        raise ValueError(f"No se extrajo texto de {path}")
    return res.text_content

texts = {
    "Attention Is All You Need": ("attention_paper.pdf", load_pdf(DOCS_DIR/"attention_paper.pdf")),
    "Retrieval-Augmented Generation": ("rag_paper.pdf", load_pdf(DOCS_DIR/"rag_paper.pdf")),
    "BERT": ("bert_paper.pdf", load_pdf(DOCS_DIR/"bert_paper.pdf")),
}
for title,(src,t) in texts.items():
    print(f"{title}: {len(t):,} chars")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def to_docs(text, title, source):
    sp = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120,
                                        separators=["\n\n","\n",".", " "])
    out=[]
    for i,ch in enumerate(sp.split_text(text)):
        if len(ch.strip())<30: continue
        out.append(Document(page_content=ch, metadata={"title":title,"source":source,"chunk_index":i}))
    return out

all_docs=[]
for title,(src,t) in texts.items():
    d=to_docs(t,title,src); all_docs+=d
    print(f"{title}: {len(d)} chunks")
print("TOTAL:", len(all_docs), "chunks")


In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))
print("dim embeddings:", EMBEDDING_DIM)


---
## C1 — FAISS vs HNSW: el compromiso `ef` ↔ recall/velocidad

Construyo dos índices FAISS sobre **los mismos vectores**:
- `IndexFlatIP` → búsqueda **exacta** (fuerza bruta), el *ground truth* de vecinos.
- `IndexHNSWFlat` → búsqueda **aproximada** navegando el grafo.

Luego, variando `efSearch`, mido el **recall@k** del HNSW contra el exacto y el
**tiempo medio** por consulta. Se espera: más `ef` → más recall pero más lento.


In [ ]:
import faiss

# 1) Vectores de todos los chunks (una sola vez)
doc_texts = [d.page_content for d in all_docs]
doc_vecs = np.array(embeddings.embed_documents(doc_texts), dtype="float32")
faiss.normalize_L2(doc_vecs)            # coseno via producto interno
n, dim = doc_vecs.shape
print(f"{n} vectores de dim {dim}")

# 2) Indice EXACTO (ground truth de vecinos)
flat = faiss.IndexFlatIP(dim)
flat.add(doc_vecs)

# 3) Indice HNSW aproximado
hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)  # M=32
hnsw.hnsw.efConstruction = 80
hnsw.add(doc_vecs)

# 4) Consultas de prueba
queries = [
    "What is scaled dot-product attention?",
    "How does RAG retrieve documents?",
    "What is masked language modeling in BERT?",
    "What optimizer trains the Transformer?",
    "What is the difference between RAG-Token and RAG-Sequence?",
]
qv = np.array(embeddings.embed_documents(queries), dtype="float32")
faiss.normalize_L2(qv)

K = 5
_, gt_idx = flat.search(qv, K)          # vecinos exactos

print(f"{'efSearch':>8} | {'recall@5':>9} | {'ms/consulta':>12}")
for ef in [8, 16, 32, 64, 128]:
    hnsw.hnsw.efSearch = ef             # ef >= k siempre
    t0 = time.perf_counter()
    _, ap_idx = hnsw.search(qv, K)
    dt = (time.perf_counter() - t0) / len(queries) * 1000
    # recall@k = fraccion de vecinos exactos recuperados por HNSW
    rec = np.mean([len(set(ap_idx[i]) & set(gt_idx[i]))/K for i in range(len(queries))])
    print(f"{ef:>8} | {rec:>9.2f} | {dt:>12.2f}")

print("\nConclusion C1: al subir efSearch, recall@5 sube hacia 1.0 pero el tiempo por")
print("consulta crece. ef debe ser >= k. La fuerza bruta (flat) es exacta pero O(N).")


---
## C2 — Tercer paper: pregunta que conecta los tres

Con BERT ya indexado, hago una pregunta que exige conectar **RAG** (usa encoders
BERT en su retriever DPR) con **BERT** (qué es) y con **Attention** (BERT y el
generador de RAG se basan en Transformers). Construyo un índice Qdrant con los
tres papers y muestro de qué papers vienen los fragmentos recuperados.


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

client = QdrantClient(":memory:")
COLLECTION = "three_papers"
client.create_collection(COLLECTION, vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
vs = QdrantVectorStore(client=client, collection_name=COLLECTION, embedding=embeddings)
vs.add_documents(all_docs)

llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=800)
retriever = vs.as_retriever(search_type="similarity_score_threshold",
                            search_kwargs={"k": 6, "score_threshold": 0.2})

def format_docs(docs):
    return "\n\n---\n\n".join(f"[{d.metadata['title']} | {d.metadata['source']}]\n{d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system","Answer using only the supplied academic-paper excerpts. If not enough evidence, say so. "
              "Answer in the language of the question. Mention the paper titles that support the answer."),
    ("human","CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())

q3 = ("The RAG paper's retriever (DPR) uses BERT encoders, and BERT is built on the Transformer. "
      "Explain how these three papers connect: what role does BERT play inside RAG, "
      "and what architecture do both BERT and the RAG generator rely on?")
ans = rag_chain.invoke(q3)
docs = retriever.invoke(q3)
print("Q:", q3, "\n")
print("A:", ans, "\n")
print("Fuentes recuperadas:")
for d in docs:
    print(f"  [{d.metadata['title']}] {d.metadata['source']} #chunk {d.metadata['chunk_index']}")
print("\nEsperado: aparecen los TRES papers y la respuesta explica que BERT provee los")
print("encoders del retriever DPR, y que BERT y el generador de RAG se basan en Transformers.")


---
## C3 — Filtros por metadata

Qdrant guarda la metadata junto al vector, así que puedo **restringir la
búsqueda a un paper concreto** sin reconstruir el índice. Comparo una búsqueda
sin filtro contra la misma búsqueda filtrada por `source = attention_paper.pdf`.


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

q = "What is multi-head attention?"

print("=== Sin filtro (todos los papers) ===")
for d in vs.similarity_search(q, k=4):
    print(f"  [{d.metadata['source']}] {d.page_content[:70].strip()}...")

print("\n=== Filtrado: solo attention_paper.pdf ===")
flt = Filter(must=[FieldCondition(key="metadata.source", match=MatchValue(value="attention_paper.pdf"))])
for d in vs.similarity_search(q, k=4, filter=flt):
    print(f"  [{d.metadata['source']}] {d.page_content[:70].strip()}...")

print("\nConclusion C3: el filtro por metadata restringe los resultados al paper pedido.")
print("Esto resuelve el problema del FAISSVectorStore plano, que no filtra por payload.")


---
## C4 — Evaluación automática (keyword-recall)

Métrica automática simple y reproducible: para cada pregunta con *ground truth*
defino las **palabras clave** que la respuesta debe contener y mido qué
fracción aparece en la respuesta del RAG (`keyword-recall`). Reporto el promedio.
Es una métrica *lexical* (no perfecta, pero objetiva y sin coste extra de API).


In [ ]:
eval_set = [
    {"q": "What BLEU score did the Transformer big model reach on WMT14 English-German?",
     "keywords": ["28.4"]},
    {"q": "How many attention heads does the base Transformer use?",
     "keywords": ["8", "head"]},
    {"q": "What retriever does RAG use?",
     "keywords": ["DPR", "dense"]},
    {"q": "What are the two RAG variants?",
     "keywords": ["sequence", "token"]},
    {"q": "What training objective does BERT use?",
     "keywords": ["masked", "language"]},
]

def keyword_recall(answer, keywords):
    a = answer.lower()
    hits = sum(1 for k in keywords if k.lower() in a)
    return hits / len(keywords)

scores = []
for item in eval_set:
    ans = rag_chain.invoke(item["q"])
    r = keyword_recall(ans, item["keywords"])
    scores.append(r)
    print(f"[recall={r:.2f}] {item['q']}")
    print(f"   A: {ans[:160].strip()}...\n")

print(f"=== keyword-recall promedio: {np.mean(scores):.2f} ===")
print("Conclusion C4: metrica automatica y reproducible del desempeno del RAG.")
print("Permite detectar regresiones al cambiar hiperparametros o el modelo.")


---
## Conclusiones de los retos

- **C1:** HNSW aproxima el vecino exacto; `efSearch` controla el compromiso
  recall↔latencia y debe cumplir `ef ≥ k`. La fuerza bruta es exacta pero no
  escala.
- **C2:** añadir un tercer paper (BERT) permite respuestas que **sintetizan tres
  fuentes**; el retriever recupera chunks de los tres papers.
- **C3:** los **filtros por metadata** de Qdrant acotan la búsqueda a un paper
  sin reconstruir el índice — algo imposible con un FAISS plano.
- **C4:** una métrica **automática** (keyword-recall) da una medida objetiva y
  barata del desempeño, útil para CI y para comparar configuraciones.
